# 策略迭代算法（Policy Iteration）

## 一、导入依赖库

In [1]:
import numpy as np     # 导入 NumPy 库，用于高性能数值计算（多维数组、矩阵运算等）
import random          # 导入 Python 内置 random 模块，可用于生成随机整数（如随机种子）
import importlib.util  # 导入 importlib.util，用于通过文件路径动态加载模块（解决文件名含特殊字符无法直接 import 的问题）

# 文件名 "01.Env_GridWorldForVI&PI.py" 含 & 符号和数字开头，不符合 Python 模块标识符规则，
# 因此使用 importlib.util.spec_from_file_location 按路径加载，与 notebook 位于同一目录
_spec = importlib.util.spec_from_file_location(
    "Env_GridWorldForVI_PI",          # 模块名（任意合法 Python 标识符，仅用于内部注册）
    "01.1.Env_GridWorldForVI&PI.py"   # 相对路径；Jupyter 运行时 cwd 默认为 notebook 所在目录
)
GridWorld_v1 = importlib.util.module_from_spec(_spec)  # 创建模块对象；类型：types.ModuleType
_spec.loader.exec_module(GridWorld_v1)                  # 执行模块代码，完成加载（等价于 import）
del _spec                                               # 删除临时变量，保持命名空间整洁

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\15513\\Desktop\\ReinforcementLearning\\01.Env_GridWorldForVI&PI.py'

## 二、初始化环境与变量

In [2]:
gamma = 0.9   # 折扣因子（Discount Factor），取值范围 (0,1]；
              # 越接近 0，agent 越"短视"（偏重即时奖励）；
              # 越接近 1，agent 越"远视"（更重视长期累积回报）

rows = 5      # 网格世界的行数，纵轴为 x 方向；需与 desc 列表长度保持一致
columns = 5   # 网格世界的列数，横轴为 y 方向；需与 desc 中每个字符串的长度保持一致

# 实例化 GridWorld 环境，各参数含义：
# - forbiddenAreaScore（float）：进入障碍格（🚫）的即时惩罚奖励，此处为 -10
# - score（float）：到达目标格（✅）的即时正奖励，此处为 1
# - desc（list[str]）：字符串列表描述地图；'.' 普通格，'#' 障碍格，'T' 目标格
# 返回：GridWorld_v1.GridWorld_v1 对象，封装环境状态、奖励函数和状态转移规则
gridworld = GridWorld_v1.GridWorld_v1(
    forbiddenAreaScore=-10,
    score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."]
)

gridworld.show()  # 打印网格地图，⬜️ 普通格，🚫 障碍格，✅ 目标格

# 初始化状态价值函数 V(s)（State-Value Function）
# - 形状：(rows * columns,) = (25,)，一维浮点数组，全零初始化
# - 语义：value[i] 表示从状态 i 出发、遵循当前策略能获得的期望累积折扣回报估计
value = np.zeros(rows * columns)

# 初始化动作价值表 Q(s,a)（Q-table / Action-Value Table）
# - 形状：(rows * columns, 5) = (25, 5)，二维浮点矩阵，全零初始化
# - 语义：qtable[i][j] 表示在状态 i 下执行动作 j 的期望累积折扣回报
# - 5 个动作分别对应：上、右、下、左、原地（由 GridWorld 环境定义）
qtable = np.zeros((rows * columns, 5))

# 基于全零 Q 表生成初始贪心策略（Greedy Policy）
# - np.argmax(qtable, axis=1)：对每行（每个状态）取动作值最大的索引
# - 返回形状：(25,)，dtype=int64；全零 Q 表时所有状态均指向动作 0
policy = np.argmax(qtable, axis=1)

gridworld.showPolicy(policy)  # 用方向箭头可视化初始策略（全部指向动作 0 方向）

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
⬆️⬆️⬆️⬆️⬆️
⬆️⏫️⏫️⬆️⬆️
⬆️⬆️⏫️⬆️⬆️
⬆️⏫️✅⏫️⬆️
⬆️⏫️⬆️⬆️⬆️


## 三、随机策略初始化

In [3]:
# 用随机整数重新初始化策略，为策略迭代提供随机起点，验证算法对任意初始策略的收敛性
# np.random.randint(low, high, size)：生成 [low, high) 范围内均匀分布的随机整数数组
# - low=0：动作编号最小值（含），对应第一个合法动作
# - high=5：动作编号上限（不含），共有 5 个动作（上/右/下/左/原地）
# - size=(rows * columns)：数组长度为 25，每个元素对应一个状态的随机初始动作
# 返回形状：(25,)，dtype=int64
# 注意：此处策略为确定性策略（每个状态 100% 选择某一动作）；
# 严格的随机策略（Stochastic Policy）应为概率分布（如 π(a|s) 为各动作的概率），
# 但为简化代码实现与可视化，此处仅考虑确定性策略
policy = np.random.randint(0, 5, size=(rows * columns))

In [4]:
gridworld.showPolicy(policy)  # 打印随机初始化后的策略，用方向箭头展示每个状态的随机初始动作选择

⬆️⬇️⬇️➡️🔄
➡️⏩️⏬⬇️⬇️
⬅️⬇️⏫️⬅️⬆️
🔄⏬✅⏫️🔄
➡️🔄➡️➡️⬆️


## 四、策略迭代算法实现

> #### 策略迭代的核心思路
>
> 策略迭代将"求最优策略"拆解为两步交替执行：
>
> **① 策略评估（Policy Evaluation）—— 基于贝尔曼方程**
>
> **作用：** 回答"当前策略有多好"——将 $V^\pi$ 从模糊的初始估计精确化为当前策略的真实价值，为下一步改进提供可靠依据。
>
> 固定当前策略 $\pi$，反复用贝尔曼方程迭代，直到状态价值收敛为 $V^\pi$：
>
> $$V^\pi(s) \leftarrow r + \gamma \cdot V^\pi(s')$$
>
> **② 策略提升（Policy Improvement）—— 借用贝尔曼最优方程的思想**
>
> **作用：** 回答"能不能做得更好"——利用已知的 $V^\pi$ 计算每个动作的收益，贪心地换成更优动作，保证新策略不劣于旧策略。
>
> 用 $V^\pi$（而非最优 $V^*$）计算动作价值，贪心选取新策略：
>
> $$Q(s,a) = r + \gamma \cdot V^\pi(s') \xrightarrow{\text{贪心}} \pi'(s) = \arg\max_a\, Q(s,a)$$
>
> 注意：此处并非直接求解贝尔曼最优方程，而是借用其"取最大值"的结构改进策略。两步交替迭代，$\pi \to \pi' \to \pi'' \to \cdots$，由策略提升定理保证单调不变差，有限步内收敛至最优策略 $\pi^*$，此时 $V^\pi = V^*$，同时满足贝尔曼最优方程。

> ### 贝尔曼方程 vs 贝尔曼最优方程
>
> #### 贝尔曼方程（Bellman Expectation Equation）
>
> 针对**特定策略 $\pi$** 下的动作价值函数，描述当前动作价值与后继状态价值之间的递推关系。
>
> **动作价值函数（随机环境通用形式）：**
>
> $$Q^\pi(s,a) = \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma \sum_{a'} \pi(a'|s') Q^\pi(s',a') \right]$$
>
> 其中后继状态价值 $V^\pi(s') = \sum_{a'} \pi(a'|s') Q^\pi(s',a')$ 是对策略 $\pi$ 的**加权期望**。
>
> **确定性环境化简形式：**
>
> $$Q^\pi(s,a) = r + \gamma \cdot V^\pi(s'), \quad V^\pi(s') = \sum_{a'} \pi(a'|s') Q^\pi(s',a')$$
>
> 含义：在策略 $\pi$ 下，动作价值 = 即时奖励 + 折扣后（按策略加权平均的后继状态价值）。
>
> ---
>
> #### 贝尔曼最优方程（Bellman Optimality Equation）
>
> 针对**最优策略 $\pi^*$** 下的最优值函数，在每一步都取**使价值最大的动作**，而不是按策略加权平均。
>
> **随机环境通用形式：**
>
> $$Q^*(s,a) = \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma \max_{a'} Q^*(s',a') \right]$$
>
> **确定性环境化简形式（本 GridWorld 所用）：**
>
> 执行动作后 $s'$ 唯一确定，$P(s'|s,a)=1$，期望消去，且 $V^*(s') = \max_{a'} Q^*(s',a')$，代入得：
>
> $$\boxed{Q(s,a) = r + \gamma \cdot V(s')}$$
>
> 这正是 `01.2.ValueIteration.ipynb` 中值迭代代码里 `qtable[i][j] = score + gamma * value[nextState]` 所实现的公式。
>
> ---
>
> #### 核心区别对比
>
> | 对比维度 | 贝尔曼方程 | 贝尔曼最优方程 |
> |---|---|---|
> | **针对对象** | 给定策略 $\pi$，评估它有多好 | 寻找最优策略 $\pi^*$ |
> | **动作选择** | $\sum_a \pi(a\|s)(\cdots)$ 按策略加权 | $\max_a(\cdots)$ 取最优动作 |
> | **$V(s')$ 含义** | 当前策略 $\pi$ 下的状态价值 | 最优状态价值 $V^*(s')$ |
> | **线性性** | **线性方程组**，可直接求解 | **非线性方程组**，需迭代求解 |
> | **对应算法** | 策略评估（Policy Evaluation） | 值迭代 / 策略迭代收敛后 |
>
> ---
>
> #### 关系总结
>
> ```
> 贝尔曼方程（对当前策略求期望）
>     ↓ 策略评估：反复更新 V(s) 直到收敛
>     ↓ 策略提升：贪心选 argmax Q(s,a)
>     ↓ 两步交替迭代 → 策略迭代算法
> 贝尔曼最优方程的解 V*(s)，Q(s,a) = r + γ·V*(s')
>     ↓ 贪心提取
> 最优策略 π*(a|s) = argmax_a Q*(s,a)
> ```
>
> **一句话总结：** 两者在确定性环境中形式完全相同，均为 $Q(s,a) = r + \gamma \cdot V(s')$，区别仅在于 $V(s')$ 的计算方式——贝尔曼方程用**加权求和** $\sum_{a'}\pi(a'|s')Q^\pi(s',a')$ 评估当前策略；贝尔曼最优方程用 $\max_{a'}Q^*(s',a')$ 直接取最优动作。

> #### 贝尔曼方程与贝尔曼最优方程的状态价值均由动作价值得出
>
> 无论贝尔曼方程还是贝尔曼最优方程，**状态价值 $V(s')$ 始终是从动作价值 $Q(s',a')$ 计算得到的**，区别仅在于聚合方式：
>
> $$V(s') = \begin{cases} \displaystyle\sum_{a'} \pi(a'|s') \cdot Q^\pi(s',a') & \text{贝尔曼方程：按策略概率加权求和} \\[6pt] \displaystyle\max_{a'} Q^*(s',a') & \text{贝尔曼最优方程：取最大值} \end{cases}$$
>
> 因此两个方程在确定性环境中可以统一写成同一形式：
>
> $$\boxed{Q(s,a) = r + \gamma \cdot V(s')}$$
>
> 这也是为何用**动作价值版本**展示两者更直观——形式完全对称，差异仅在 $V(s')$ 的定义上一目了然。

### 4.1 截断策略评估版（Truncated Policy Iteration）

In [ ]:
gridworld.show()              # 打印网格地图，确认当前环境配置
gridworld.showPolicy(policy)  # 打印当前随机初始策略的可视化
print("random policy")        # 输出标签，说明此处展示的是随机初始策略

# 重置状态价值函数为全零（每次运行策略迭代前需重新初始化，避免上次结果的干扰）
# 形状：(25,)，dtype=float64
value = np.zeros(rows * columns)

# 保存上一轮价值向量，初始设为 value+1，确保第一次能进入外层 while 循环
# 形状：(25,)，dtype=float64
value_pre = value.copy() + 1

cnt = 0  # 外层迭代计数器，统计策略迭代（Policy Iteration）的总轮次

# ── 外层收敛循环（策略评估 + 策略提升 交替执行）────────────────────────────────
# 收敛条件：相邻两轮价值函数的 L2 距离平方和 ≤ 0.001
# 一旦策略收敛到最优，对应的价值函数也不再变化，循环退出
while np.sum((value_pre - value) ** 2) > 0.001:

    # ── 策略评估（Policy Evaluation）──────────────────────────────────────────
    value_pre = value.copy()  # 记录本轮开始前的价值向量，供外层收敛判断使用

    # value0：内层贝尔曼迭代的"旧值"缓存，初始设为 value+1 以触发内层循环
    # 形状：(25,)，dtype=float64
    value0 = value.copy() + 1

    # truncatedCnt：截断策略评估（Truncated Policy Evaluation）的最大内层迭代次数
    # 控制每轮策略评估执行多少步贝尔曼更新，影响外层收敛速度：
    # 取值 1→约 50 次外层迭代；2→26 次；3→18 次；4→14 次；10→6 次；100→2 次
    truncatedCnt = 10

    # ── 内层贝尔曼迭代（在当前固定策略下求解价值函数）──────────────────────────
    # 收敛条件：价值变化量 ≤ 0.001，或已达最大截断次数
    while np.sum((value0 - value) ** 2) > 0.001:

        value0 = value.copy()  # 保存本次迭代开始前的价值向量（用于下一次收敛判断）

        truncatedCnt = truncatedCnt - 1  # 消耗一次内层迭代机会（截断次数递减）
        if truncatedCnt < 0:             # 超过最大截断次数则提前退出内层循环
            break

        # 遍历所有状态，使用当前固定策略 policy 执行单步贝尔曼期望更新
        for i in range(rows * columns):  # i：状态编号，范围 [0, 24]

            j = policy[i]  # 取当前确定性策略在状态 i 下指定的动作编号（int）

            # getScore(state: int, action: int) → (reward: float, next_state: int)
            # - score（float）：在状态 i 执行动作 j 后获得的即时奖励 r
            # - nextState（int）：执行动作后转移到的下一状态编号
            score, nextState = gridworld.getScore(i, j)

            # ── 贝尔曼方程的状态价值在固定确定性策略下的化简过程 ────────────
            # 第一步：状态价值由动作价值加权求和得到（通用形式）
            #   V^π(s) = Σ_a π(a|s)·Q^π(s,a)
            #   其中 Q^π(s,a) = Σ_{s'} P(s'|s,a)·[r + γ·V^π(s')]
            # 第二步：固定确定性策略，π(a|s) = 1（a == j），其余为 0，Σ_a 消去
            #   V^π(s) = Q^π(s,j) = Σ_{s'} P(s'|s,j)·[r + γ·V^π(s')]
            # 第三步：确定性环境，执行动作 j 后 s' 唯一确定，P(s'|s,j)=1，Σ_{s'} 消去
            #   V^π(s) ← r + γ · V^π(s')
            # 其中 V(s') 取自上一步保存的 value0（旧值），保证同步更新的稳定性
            # ── 为何策略固定仍需反复迭代 ────────────────────────────────────
            # V(s) 的定义是递归的：当前状态价值依赖邻居价值，邻居又依赖其邻居
            # 初始 value 全为 0，并非真实价值；每轮迭代将价值信息向外扩散一步：
            #   第 1 轮：紧邻目标的状态获得准确值
            #   第 2 轮：再外一层的状态根据邻居更新
            #   ……直到所有状态收敛到真实 V^π（策略评估完成）
            value[i] = score + value0[nextState] * gamma

    # ── 策略提升（Policy Improvement）────────────────────────────────────────
    # 基于最新 value，重新计算所有状态下所有动作的动作价值 Q(s,a)
    for i in range(rows * columns):  # 遍历所有状态

        for j in range(5):           # 遍历 5 个动作

            # getScore(state, action) → (reward: float, next_state: int)
            score, nextState = gridworld.getScore(i, j)

            # Q(s,a) = r + γ · V(s')，利用当前最新 value 估计每个动作的价值
            # qtable[i][j] 为标量 float
            qtable[i][j] = score + gamma * value[nextState]

    # 贪心策略更新：对每个状态选取 Q 值最大的动作作为新策略（策略提升步骤）
    # np.argmax(qtable, axis=1)：沿动作维度取最大值索引，形状：(25,)，dtype=int64
    policy = np.argmax(qtable, axis=1)

    cnt = cnt + 1                                                    # 外层迭代轮次加 1
    print(f"\n{'='*20} 第 {cnt} 轮策略迭代 {'='*20}")               # 分隔线 + 轮次标题
    print("[策略]")                                                   # 策略区块标题
    gridworld.showPolicy(policy)                                      # 打印本轮策略提升后的新策略可视化
    print("[状态价值矩阵]")                                            # 价值区块标题
    print(np.round(value.reshape(rows, columns), 1))                 # 打印本轮价值函数矩阵（5×5，保留 1 位小数）

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
⬆️⬇️⬇️➡️🔄
➡️⏩️⏬⬇️⬇️
⬅️⬇️⏫️⬅️⬆️
🔄⏬✅⏫️🔄
➡️🔄➡️➡️⬆️
random policy
🔄⬅️➡️➡️⬇️
⬆️⏫️⏩️⬆️⬆️
⬇️⬅️⏬➡️⬆️
🔄⏪✅⏩️⬆️
⬆️⏩️➡️➡️⬆️
[[ -6.5 -65.1 -65.1   0.    0. ]
 [-65.1 -65.1 -65.1 -55.1   0. ]
 [ -6.5 -65.1 -65.1 -65.1   0. ]
 [  0.  -65.1 -65.1 -55.1   0. ]
 [-65.1 -65.1   0.    0.    0. ]]
1
➡️➡️➡️➡️⬇️
⬇️⏬⏫️⬆️⬆️
➡️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️➡️⬆️
[[-2.3 -2.3  0.   0.   0. ]
 [-2.3 -2.3  0.   0.   0. ]
 [ 0.   0.   1.   0.   0. ]
 [ 0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0. ]]
2
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬆️
[[0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  6.5 0.  0. ]
 [0.  6.5 6.5 6.5 0. ]
 [0.  5.5 6.5 0.  0. ]]
3
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬅️
[[0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  8.8 0.  0. ]
 [0.  8.8 8.8 8.8 0. ]
 [0.  7.8 8.8 7.8 0. ]]
4
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
[[0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  9.6 0.  0. ]
 [0.  9.6 9.6 9.6

### 4.2 重置重跑（Policy Iteration Reset）

In [6]:
# 保留上一轮策略迭代得到的最终策略 policy（已接近最优），仅重置价值函数重新运行，
# 目的：对比从"近似最优策略"出发时，策略迭代能更快收敛（外层迭代次数明显减少）
gridworld.show()              # 打印网格地图，确认环境配置
gridworld.showPolicy(policy)  # 打印当前策略（上一段代码收敛后的最优策略）
print("random policy")        # 输出标签（注意：此时 policy 并非随机，而是上一轮的优化结果）

# 重置状态价值函数，从全零重新开始策略评估
# 形状：(25,)，dtype=float64
value = np.zeros(rows * columns)

# 保存上一轮价值向量，初始设为 value+1，确保能进入外层 while 循环
# 形状：(25,)，dtype=float64
value_pre = value.copy() + 1

cnt = 0  # 外层迭代计数器

# ── 外层收敛循环（策略评估 + 策略提升 交替执行）────────────────────────────────
# 收敛条件：相邻两轮价值函数的 L2 距离平方和 ≤ 0.001
while np.sum((value_pre - value) ** 2) > 0.001:

    # ── 策略评估（Policy Evaluation）──────────────────────────────────────────
    value_pre = value.copy()  # 记录本轮开始前的价值向量，供外层收敛判断使用

    # value0：内层贝尔曼迭代的"旧值"缓存
    # 形状：(25,)，dtype=float64
    value0 = value.copy() + 1

    # 截断策略评估的最大内层迭代次数，与上一实现相同
    truncatedCnt = 10

    # ── 内层贝尔曼迭代（固定策略下求解价值函数）──────────────────────────────
    # 收敛条件：价值变化量 ≤ 0.001，或已达最大截断次数
    while np.sum((value0 - value) ** 2) > 0.001:

        value0 = value.copy()  # 保存旧值，供下次收敛判断使用

        truncatedCnt = truncatedCnt - 1  # 消耗一次截断计数
        if truncatedCnt < 0:             # 超过最大截断次数则退出内层
            break

        # 遍历所有状态，用贝尔曼方程的状态价值更新 V(s)
        for i in range(rows * columns):  # i：状态编号，范围 [0, 24]

            j = policy[i]  # 取当前确定性策略在状态 i 下的动作（int）

            # getScore(state: int, action: int) → (reward: float, next_state: int)
            score, nextState = gridworld.getScore(i, j)

            # 贝尔曼方程的状态价值化简形式：V(s) ← r + γ · V(s')
            # 使用旧值 value0 保证同步更新稳定性；策略固定但仍需反复迭代，
            # 因为 V(s) 递归依赖邻居价值，每轮只能向外扩散一步直至全局收敛
            # 确定性策略下 V^π(s) = Q^π(s,π(s))，故可跳过 Q 表直接更新
            value[i] = score + value0[nextState] * gamma

    # ── 策略提升（Policy Improvement）────────────────────────────────────────
    # 遍历所有状态和动作，重新计算动作价值 Q(s,a) = r + γ · V(s')
    for i in range(rows * columns):  # 遍历所有状态

        for j in range(5):           # 遍历 5 个动作

            # getScore(state, action) → (reward: float, next_state: int)
            score, nextState = gridworld.getScore(i, j)

            # 更新动作价值表：Q(s,a) = r + γ · V(s')
            # qtable[i][j] 为标量 float
            qtable[i][j] = score + gamma * value[nextState]

    # 贪心策略更新：选取 Q 值最大的动作作为新策略
    # np.argmax(qtable, axis=1)：返回形状 (25,)，dtype=int64
    policy = np.argmax(qtable, axis=1)

    cnt = cnt + 1                                                    # 外层迭代轮次加 1
    print(f"\n{'='*20} 第 {cnt} 轮策略迭代 {'='*20}")               # 分隔线 + 轮次标题
    print("[策略]")                                                   # 策略区块标题
    gridworld.showPolicy(policy)                                      # 打印本轮策略提升后的新策略可视化
    print("[状态价值矩阵]")                                            # 价值区块标题
    print(np.round(value.reshape(rows, columns), 1))                 # 打印本轮价值函数矩阵（5×5，保留 1 位小数）

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️➡️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
random policy
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️➡️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
[[0.         0.38742049 0.8178877  1.2961846  1.8276256 ]
 [0.         0.         1.2961846  1.8276256  2.4181156 ]
 [0.         0.         6.5132156  2.4181156  3.0742156 ]
 [0.         6.5132156  6.5132156  6.5132156  3.8032156 ]
 [0.         5.5132156  6.5132156  5.5132156  4.6132156 ]]
1
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️➡️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
[[2.27101786 2.65843834 3.08890555 3.56720245 4.09864345]
 [1.92233941 2.27101786 3.56720245 4.09864345 4.68913345]
 [1.60852882 1.32609928 8.78423345 4.68913345 5.34523345]
 [1.32609928 8.78423345 8.78423345 8.78423345 6.07423345]
 [1.0719127  7.78423345 8.78423345 7.78423345 6.88423345]]
2
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️➡️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
[[3.06287282 3.45029331 3.88076052 4.35905742 4.89049842]
 [2.71419438 3.06287282 4.35905742 4.89049842 5.48098842]
 [2.40038378 2.11795425 9.57